# Experiment: Skimage Memorization Sanity

This notebook is a hands-on sanity check for the canonical memorization metrics in `mp_metrics`.
We use a few real grayscale images from `skimage.data`, treat one image as the "generated source",
include that same image inside the reference set, add increasing Gaussian noise, and watch how
`f_mem` and the mean nearest-neighbor ratio change.

The expected behavior is simple:
- at `sigma = 0`, memorization should be maximal because the sample is an exact copy of a reference image
- as noise increases, `f_mem` should fall and the mean ratio `r = d1 / d2` should rise
- changing the threshold should move the `f_mem` curves in a transparent, controllable way


## What This Notebook Covers

- a direct tensor-vs-tensor use of `compute_memorization_metrics`
- a real-image sanity check built from `skimage.data`
- editable parameters for image set, threshold, noise range, and number of noisy draws
- plots for `f_mem` and mean NN ratio across noise levels
- a visual panel of noisy samples so the metric curves stay grounded in actual inputs


In [ ]:
# Setup: imports, repo path, and reproducibility
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch


def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() and (candidate / "src").exists():
            return candidate
        if (candidate / "src").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the repo root. Start Jupyter from somewhere inside the repo tree."
    )


repo_root = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(repo_root / "src"))

from mp_metrics import compute_memorization_metrics

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
SEED


In [ ]:
# Parameters: edit these first
image_names = ["camera", "coins", "moon", "page"]
image_size = 128
n_levels = 12
sigma_min = 0.0
sigma_max = 0.50
samples_per_level = 64
ratio_threshold = 0.50
chunk_size = 256

sigma_values = np.linspace(sigma_min, sigma_max, n_levels)

{
    "image_names": image_names,
    "image_size": image_size,
    "n_levels": n_levels,
    "sigma_values": sigma_values.round(3).tolist(),
    "samples_per_level": samples_per_level,
    "ratio_threshold": ratio_threshold,
}


## Helper Functions

These are lightweight notebook-local helpers, kept close to the script version in
`reference/sanity/sanity_skimage_mem_curves.py`. The metric call itself stays canonical:
`compute_memorization_metrics(...)` comes from `mp_metrics`.


In [ ]:
from skimage import color, data, transform, util


def load_skimage_image(name, size):
    if not hasattr(data, name):
        raise ValueError(f"Unknown skimage.data image: {name}")

    image = getattr(data, name)()
    image = util.img_as_float32(image)

    if image.ndim == 3:
        image = color.rgb2gray(image)

    image = transform.resize(
        image,
        (size, size),
        anti_aliasing=True,
        preserve_range=True,
    ).astype(np.float32)
    image = np.clip(image, 0.0, 1.0)
    return torch.from_numpy(image).unsqueeze(0)  # [1, H, W]


def build_reference_set(names, size):
    tensors = [load_skimage_image(name, size) for name in names]
    return torch.stack(tensors, dim=0)  # [N, 1, H, W]


def noisy_batch(base_image, sigma, n_samples, generator=None):
    noise = torch.randn(
        n_samples,
        *base_image.shape,
        generator=generator,
        dtype=base_image.dtype,
    )
    noisy = base_image.unsqueeze(0) + sigma * noise
    return noisy.clamp(0.0, 1.0)


def summarize_source_against_reference(base_image, reference, sigma_values, samples_per_level, ratio_threshold, chunk_size, seed):
    generator = torch.Generator().manual_seed(seed)
    rows = []
    for sigma in sigma_values:
        samples = noisy_batch(base_image, float(sigma), samples_per_level, generator=generator)
        metrics = compute_memorization_metrics(
            samples,
            reference,
            ratio_threshold=ratio_threshold,
            chunk_size=chunk_size,
        )
        rows.append(
            {
                "sigma": float(sigma),
                "f_mem": metrics["f_mem"],
                "mean_nn_ratio": metrics["mean_nn_ratio"],
                "mean_d1": metrics["mean_d1"],
                "mean_d2": metrics["mean_d2"],
            }
        )
    return rows


In [ ]:
# Build and preview the reference set
reference = build_reference_set(image_names, image_size)

fig, axes = plt.subplots(1, len(image_names), figsize=(3 * len(image_names), 3.2), constrained_layout=True)
if len(image_names) == 1:
    axes = [axes]

for ax, name, tensor in zip(axes, image_names, reference):
    ax.imshow(tensor.squeeze(0), cmap="gray", vmin=0.0, vmax=1.0)
    ax.set_title(name)
    ax.axis("off")

plt.show()


## Compute Memorization Curves

For each image in the reference set:
- treat that image as the clean prototype
- generate noisy copies of it at increasing `sigma`
- compare those samples against the full reference set, including the original image itself


In [ ]:
curve_data = {}

for idx, name in enumerate(image_names):
    base = reference[idx]
    curve_data[name] = summarize_source_against_reference(
        base_image=base,
        reference=reference,
        sigma_values=sigma_values,
        samples_per_level=samples_per_level,
        ratio_threshold=ratio_threshold,
        chunk_size=chunk_size,
        seed=SEED + idx,
    )

summary_rows = []
for name, rows in curve_data.items():
    summary_rows.append(
        {
            "image": name,
            "f_mem@min_sigma": rows[0]["f_mem"],
            "f_mem@max_sigma": rows[-1]["f_mem"],
            "mean_r@min_sigma": rows[0]["mean_nn_ratio"],
            "mean_r@max_sigma": rows[-1]["mean_nn_ratio"],
        }
    )

summary_rows


In [ ]:
# Plot the memorization curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

for name, rows in curve_data.items():
    sigmas = [row["sigma"] for row in rows]
    f_mem = [row["f_mem"] for row in rows]
    mean_r = [row["mean_nn_ratio"] for row in rows]

    axes[0].plot(sigmas, f_mem, marker="o", linewidth=1.6, label=name)
    axes[1].plot(sigmas, mean_r, marker="o", linewidth=1.6, label=name)

axes[0].set_title("f_mem vs Noise Sigma")
axes[0].set_xlabel("sigma")
axes[0].set_ylabel("f_mem")
axes[0].set_ylim(-0.02, 1.02)
axes[0].grid(alpha=0.3)

axes[1].set_title("Mean NN Ratio vs Noise Sigma")
axes[1].set_xlabel("sigma")
axes[1].set_ylabel("mean(d1 / d2)")
axes[1].grid(alpha=0.3)
axes[1].legend(loc="best", fontsize=8)

fig.suptitle(
    f"skimage memorization sanity | threshold={ratio_threshold:.2f} | samples/level={samples_per_level}"
)
plt.show()


## Visual Sanity Panel

This keeps the metric curves grounded. We look at one selected source image and show how the noisy samples
visibly degrade as `sigma` increases.


In [ ]:
# Pick one image and inspect noisy samples across the noise schedule
inspect_name = image_names[0]
inspect_sigmas = [sigma_values[0], sigma_values[len(sigma_values) // 2], sigma_values[-1]]
base = reference[image_names.index(inspect_name)]
generator = torch.Generator().manual_seed(SEED + 123)

fig, axes = plt.subplots(1, len(inspect_sigmas) + 1, figsize=(3 * (len(inspect_sigmas) + 1), 3.1), constrained_layout=True)
axes[0].imshow(base.squeeze(0), cmap="gray", vmin=0.0, vmax=1.0)
axes[0].set_title(f"{inspect_name}\nclean reference")
axes[0].axis("off")

for ax, sigma in zip(axes[1:], inspect_sigmas):
    sample = noisy_batch(base, float(sigma), n_samples=1, generator=generator)[0]
    ax.imshow(sample.squeeze(0), cmap="gray", vmin=0.0, vmax=1.0)
    ax.set_title(f"sigma={float(sigma):.3f}")
    ax.axis("off")

plt.show()


## Threshold Sweep For One Source Image

This cell shows directly how the configurable `f_mem` threshold changes the curve. The same noisy samples are
re-evaluated under several thresholds.


In [ ]:
thresholds = [0.20, 0.33, 0.50, 0.80]
base = reference[0]
threshold_curves = {}

for threshold in thresholds:
    threshold_curves[threshold] = summarize_source_against_reference(
        base_image=base,
        reference=reference,
        sigma_values=sigma_values,
        samples_per_level=samples_per_level,
        ratio_threshold=threshold,
        chunk_size=chunk_size,
        seed=SEED + 999,
    )

fig, ax = plt.subplots(figsize=(6.5, 4.2), constrained_layout=True)
for threshold, rows in threshold_curves.items():
    ax.plot(
        [row["sigma"] for row in rows],
        [row["f_mem"] for row in rows],
        marker="o",
        linewidth=1.6,
        label=f"threshold={threshold:.2f}",
    )

ax.set_title(f"Threshold sensitivity for {image_names[0]}")
ax.set_xlabel("sigma")
ax.set_ylabel("f_mem")
ax.set_ylim(-0.02, 1.02)
ax.grid(alpha=0.3)
ax.legend(loc="best")
plt.show()


## Notes

What to look for:
- `f_mem` should start near `1.0` at zero noise because the source image is present in the reference set.
- As noise grows, the nearest-neighbor ratio should increase and memorization should drop.
- Looser thresholds push the `f_mem` curves upward, which is why the threshold should always be explicit in configs,
  notebooks, and saved metric reports.

If you want to compare this notebook against the script version, see:
- `reference/sanity/sanity_skimage_mem_curves.py`
